# Extract spin density from VASP CHGCAR

This script reads the spin-polarized CHGCAR in the current working directory.
It writes SPIN_CHGCAR containing magnetization density rho_up - rho_down for VESTA.


In [ ]:
from pathlib import Path
import math

def _tokens_are_ints(tokens):
    try:
        return [int(x) for x in tokens]
    except ValueError:
        return None

def _find_counts_line(lines):
    """
    Find the atom-count line in a POSCAR/CHGCAR header.
    Supports both VASP 4 and VASP 5 style headers.
    """
    # VASP 5 style:
    # line 5: element symbols
    # line 6: atom counts
    possible_counts = _tokens_are_ints(lines[6].split())
    if possible_counts is not None:
        return 6, possible_counts
    # VASP 4 style:
    # line 5: atom counts
    possible_counts = _tokens_are_ints(lines[5].split())
    if possible_counts is not None:
        return 5, possible_counts

    raise RuntimeError("Could not identify the atom-count line in CHGCAR.")

def _find_first_grid_line(lines):
    """
    Find the first volumetric grid line after atomic coordinates.
    """
    counts_idx, counts = _find_counts_line(lines)
    n_atoms = sum(counts)

    coord_type_idx = counts_idx + 1

    # Optional Selective dynamics line
    if lines[coord_type_idx].strip().lower().startswith("s"):
        coord_type_idx += 1

    coord_start_idx = coord_type_idx + 1
    after_coords_idx = coord_start_idx + n_atoms

    for i in range(after_coords_idx, len(lines)):
        tokens = lines[i].split()
        grid = _tokens_are_ints(tokens)
        if grid is not None and len(grid) == 3 and all(x > 1 for x in grid):
            return i, grid

    raise RuntimeError("Could not find the first volumetric grid line.")

def _read_n_values(lines, start_idx, n_values):
    """
    Read n_values floating-point numbers starting after a grid line.
    """
    values = []
    i = start_idx

    while i < len(lines) and len(values) < n_values:
        values.extend(lines[i].split())
        i += 1

    if len(values) < n_values:
        raise RuntimeError("Not enough volumetric data values found.")

    return values[:n_values], i


def _find_next_same_grid(lines, start_idx, grid):
    """
    Find the next occurrence of the same volumetric grid line.
    In spin-polarized CHGCAR, this usually precedes magnetization density.
    """
    for i in range(start_idx, len(lines)):
        tokens = lines[i].split()
        candidate = _tokens_are_ints(tokens)
        if candidate == grid:
            return i

    raise RuntimeError(
        "Could not find a second volumetric data block. "
        "This CHGCAR may not contain spin density."
    )

def extract_spin_density_chgcar(input_chgcar="CHGCAR", output_chgcar="SPIN_CHGCAR"):
    """
    Extract magnetization density from a spin-polarized CHGCAR and write it as
    a CHGCAR-like file that VESTA can open directly.

    The output file contains the original structure header plus one volumetric
    data block: rho_up - rho_down.
    """
    input_chgcar = Path(input_chgcar).expanduser().resolve()
    output_chgcar = Path(output_chgcar).expanduser().resolve()

    if not input_chgcar.exists():
        raise FileNotFoundError(f"Cannot find input file: {input_chgcar}")

    lines = input_chgcar.read_text().splitlines(keepends=True)

    first_grid_idx, grid = _find_first_grid_line(lines)
    n_grid = math.prod(grid)

    # First block: total charge density
    _, after_total_idx = _read_n_values(lines, first_grid_idx + 1, n_grid)

    # Second block: magnetization density
    second_grid_idx = _find_next_same_grid(lines, after_total_idx, grid)
    spin_values, _ = _read_n_values(lines, second_grid_idx + 1, n_grid)

    spin_float = [float(x) for x in spin_values]

    with open(output_chgcar, "w") as f:
        # Keep original structure header and the first grid line
        for line in lines[:first_grid_idx + 1]:
            f.write(line)

        # Write magnetization density as the only volumetric data block
        for i in range(0, len(spin_float), 5):
            f.write(" ".join(f"{x: .11E}" for x in spin_float[i:i + 5]) + "\n")

    print("Done.")
    print(f"Working directory: {Path.cwd()}")
    print(f"Input : {input_chgcar}")
    print(f"Output: {output_chgcar}")
    print(f"Grid  : {grid[0]} x {grid[1]} x {grid[2]}")
    print(f"Spin density min = {min(spin_float): .6E}")
    print(f"Spin density max = {max(spin_float): .6E}")

    return output_chgcar

# Read ./CHGCAR and create ./SPIN_CHGCAR
spin_chgcar = extract_spin_density_chgcar()
